# Interactive SWC Compartment Selector

This notebook provides three selector components:

1. Interactive 3D SWC rendering for one neuron with clickable compartments and visible section names.
2. Apply glia-loss-like local extracellular K changes to selected compartments.
3. Use selected compartment(s) to assign AIS.

It uses helper functions from `tools/swc_interactive_selector.py`.



In [ ]:
# Optional (only needed for SELECTOR_BACKEND='widget'):
# %pip install -U anywidget ipywidgets plotly



In [ ]:
from pathlib import Path
import sys
import inspect
import importlib
import ipywidgets, plotly

try:
    import anywidget
    anywidget_version = anywidget.__version__
except Exception:
    anywidget_version = 'not installed (OK for static backend)'

print('plotly    =', plotly.__version__)
print('ipywidgets=', ipywidgets.__version__)
print('anywidget =', anywidget_version)


def _find_work_root() -> Path:
    candidates = []
    env_root = os.environ.get('DIGIFLY_VIP_GLIA_ROOT', '').strip() if 'os' in globals() else ''
    if env_root:
        candidates.append(Path(env_root))
    try:
        candidates.append(Path.cwd())
    except Exception:
        pass
    try:
        cwd = Path.cwd()
        for base in [cwd] + list(cwd.parents):
            candidates.append(base / 'Phase 2' / 'apps' / 'VIP_Glia_Sim')
            candidates.append(base / 'apps' / 'VIP_Glia_Sim')
    except Exception:
        pass
    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except Exception:
            continue
        for root in [candidate] + list(candidate.parents):
            if (root / 'tools' / 'swc_interactive_selector.py').exists():
                return root
    raise RuntimeError('Could not locate VIP_Glia_Sim app root. Set DIGIFLY_VIP_GLIA_ROOT.')


import os
WORK_ROOT = _find_work_root()
PHASE2_ROOT = Path(os.environ.get('DIGIFLY_PHASE2_ROOT', str(WORK_ROOT.parents[1]))).expanduser().resolve()

if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
if str(PHASE2_ROOT) not in sys.path:
    sys.path.insert(0, str(PHASE2_ROOT))

# Force-refresh local selector module so notebook reruns pick up file edits.
import tools.swc_interactive_selector as _selector_mod
importlib.reload(_selector_mod)

from tools.swc_interactive_selector import (
    InteractiveSWCSelector,
    apply_glia_loss_to_selected_sections,
    load_swc_cell_for_selection,
    make_glia_loss_spec_from_selection,
    section_metadata_table,
    set_ais_from_selection,
)

print('selector module file =', _selector_mod.__file__)
print('InteractiveSWCSelector.__init__ =', inspect.signature(InteractiveSWCSelector.__init__))
print('WORK_ROOT  =', WORK_ROOT)
print('PHASE2_ROOT=', PHASE2_ROOT)



In [ ]:
# User controls
SWC_DIR = (PHASE2_ROOT / 'data' / 'export_swc').resolve()
NEURON_ID = 10000

# Build an SWCCell for standalone compartment picking.
cell = load_swc_cell_for_selection(
    neuron_id=NEURON_ID,
    swc_dir=SWC_DIR,
    cfg_overrides={
        'celsius_C': 22.0,
        'ais_min_xloc': 0.05,
    },
)

print('Loaded neuron:', NEURON_ID)
print('Total sections:', len(getattr(cell, '_secs', [])))
section_metadata_table(cell).head(10)



## Part 1: Interactive 3D Compartment Selection

- Use `SELECTOR_BACKEND='static'` (default below) when the notebook frontend throws `AnyModel/anywidget` errors.
- In `static` mode, hover in 3D to inspect section names and select compartments from the **Pick** list.
- If the frontend fully supports FigureWidget, switch to `SELECTOR_BACKEND='widget'` for click-to-select directly on traces.



In [ ]:
SELECTOR_BACKEND = 'static'  # 'widget' for click-to-select when FigureWidget works
selector = InteractiveSWCSelector(
    cell,
    title=f'Neuron {NEURON_ID} compartment picker',
    backend=SELECTOR_BACKEND,
)
selector.show()



In [ ]:
import plotly, ipywidgets
print('plotly:', plotly.__version__)
print('ipywidgets:', ipywidgets.__version__)
print("If the frontend reports 'Failed to load model class AnyModel', keep SELECTOR_BACKEND='static'.")



## Part 2: Apply Glia-Loss Effect to Selected Compartments

- Select compartments in the 3D viewer first.
- Then run this cell to apply extracellular K perturbation on only those selected sections.



In [ ]:
# Example perturbation controls
TARGET_KO_mM = 6.5
TARGET_KI_mM = 140.0
UPDATE_EK = True

applied_df = apply_glia_loss_to_selected_sections(
    cell,
    selector,
    ko_mM=TARGET_KO_mM,
    ki_mM=TARGET_KI_mM,
    update_ek=UPDATE_EK,
)

print('Applied to sections:', len(applied_df))
applied_df



In [ ]:
# Optional: export selection in GLIA_LOSS_SPEC format for glia_simulation.ipynb
# (compartment uses sec:<section_name> entries)

GLIA_LOSS_SPEC = make_glia_loss_spec_from_selection(
    selector,
    ko_mM=TARGET_KO_mM,
    ki_mM=TARGET_KI_mM,
)

print('GLIA_LOSS_SPEC entries:', len(GLIA_LOSS_SPEC))
GLIA_LOSS_SPEC[:5]



## Part 3: Set AIS from Selected Compartment

- If multiple are selected, this uses last clicked section by default.
- `persist_override=True` writes an AIS override row for this neuron (used in future builds).



In [ ]:
AIS_XLOC = 0.5
PERSIST_OVERRIDE = False

ais_info = set_ais_from_selection(
    cell,
    selector,
    xloc=AIS_XLOC,
    persist_override=PERSIST_OVERRIDE,
)

print(ais_info)
selector.refresh_ais_marker()
selector.selected_table()



In [ ]:
# Optional AIS marker visual check,
# rerun the Part 1 display cell above (selector.show()).

print('Done.')

